# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step how to load, explore, and process a dataset described in Croissant format using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets (`@id`) and for each, show the fields/columns with their corresponding `@id` and name.

In [ ]:
# List all record sets and their fields by @id
print('Record Sets:')
for rs in metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}  |  Name: {rs.name if hasattr(rs, 'name') else ''}")
    print("  Fields/Columns:")
    for f in rs.fields:
        field_name = getattr(f, 'name', '')
        print(f"    - Field @id: {f.id}  |  Name: {field_name}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. The exact `@id`s for the record set and fields are required, as listed above.

Below, we load all record sets present in the dataset and display the columns of the main experimental record set.

In [ ]:
# Prepare a dictionary of DataFrames for all record sets
dataframes = {}
# List all record set ids
record_sets_ids = [rs.id for rs in metadata.record_sets]

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, pick the first record set for column exploration
main_record_set_id = record_sets_ids[0] if record_sets_ids else None
if main_record_set_id:
    print(f"Columns in record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by key attributes.

Let's select a numeric field in the main record set (e.g., coverage percentage or peptide counts), filter by a threshold, normalize it, and optionally group by a categorical field (e.g., description if present).

> **Note:** Replace the example field `"coverage"` and `"description"` below with the actual field `@id` from the column listing if they differ in your dataset.

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Example: Find a numeric field by inspecting columns
    numeric_field_id = None
    for col in df.columns:
        # Choose the first plausible numeric field
        if df[col].dtype in [np.float64, np.int64, float, int]:
            numeric_field_id = col
            break

    if not numeric_field_id:
        # Try by name
        for col in df.columns:
            if 'coverage' in col.lower() or 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower():
                numeric_field_id = col
                break

    if numeric_field_id:
        print(f"Using numeric field: {numeric_field_id}")
        # Make numeric (in case it's string)
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df[[numeric_field_id]].head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a string field, for example, 'description', 'accession' or similar
        group_field_id = None
        for col in df.columns:
            if col.lower() in ["description", "accession", "protein description"]:
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in the main record set.")
else:
    print("No main record set for EDA.")

## 5. Visualization
Visualize data distributions or field relationships in the dataset.

We provide a histogram of the selected numeric field and, if grouped data is available, a bar plot of group means.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].hist(bins=30, edgecolor='k')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if 'grouped_df' in locals() and group_field_id:
        # Show top 10 groups
        plt.figure(figsize=(10, 4))
        grouped_df.sort_values(numeric_field_id, ascending=False)[:10].plot(
            x=group_field_id, y=numeric_field_id, kind='bar', legend=False
        )
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} per {group_field_id} (Top 10)')
        plt.tight_layout()
        plt.show()
else:
    print("Visualization could not be generated (missing field or data).")

## 6. Conclusion
We loaded the Mass Spectrometry dataset via its Croissant schema, explored its structure using `mlcroissant`, and performed basic exploratory data analysis and visualization on its main record set. This approach can be extended for more advanced statistical analysis or feature engineering as required by downstream applications.